# Comparação dos resultados

Este notebook descobre todos os cenários em `tests/results`, calcula as métricas por execução e mostra cada gráfico em uma célula independente. Ele reutiliza as funções do fluxo oficial para manter métricas, ordem, nomes e cores sincronizados com `plot_comparison_aggregated.py`.

## Preparação

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

current_dir = Path.cwd().resolve()
project_root = (
    current_dir.parent.parent
    if current_dir.name == "notebooks" and current_dir.parent.name == "tests"
    else current_dir
)
results_dir = project_root / "tests/results"

if not results_dir.is_dir():
    raise FileNotFoundError("A pasta tests/results não foi encontrada.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from plot_comparison_aggregated import metric_upper_bound, prepare_plot_data
from plot_comparison_common import (
    COMPARISON_METRICS,
    CONFIGURATION_COLORS,
    CONFIGURATION_LABELS,
)
from plot_helper import compute_run_metrics, discover_result_files, summarize_runs

## Métricas por execução

A descoberta não usa uma lista local de cenários. Todo CSV no padrão `<execução>_<ordem>_<cenário>.csv` participa da comparação, incluindo HPA e VPA.

In [ ]:
discovered_df = discover_result_files(results_dir, CONFIGURATION_LABELS)
run_df = pd.DataFrame(
    compute_run_metrics(row) for _, row in discovered_df.iterrows()
).sort_values(["order", "run", "configuration"])
summary_df = summarize_runs(run_df, COMPARISON_METRICS)

availability_df = (
    run_df.groupby(["order", "configuration", "label"], as_index=False)
    .size()
    .rename(columns={"size": "execuções"})
)

print(f"Arquivos analisados: {len(run_df)}")
availability_df

## Dados calculados

In [ ]:
run_df

In [ ]:
summary_df

## Gráficos separados

A preparação visual fica nesta célula. Cada gráfico é executado em uma das células seguintes e, portanto, aparece em seu próprio bloco de saída.

In [ ]:
sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
    },
)

configurations_df, plot_df = prepare_plot_data(run_df)
configuration_order = configurations_df["configuration"].tolist()
configuration_labels = configurations_df["label"].tolist()
fallback_colors = sns.color_palette("Set2", n_colors=len(configurations_df))
configuration_palette = {
    configuration: CONFIGURATION_COLORS.get(configuration, fallback_colors[index])
    for index, configuration in enumerate(configuration_order)
}
mean_df = plot_df.groupby(
    ["metric", "configuration"], as_index=False, observed=False
)["value"].mean()

stat_legend = [
    Line2D([0], [0], color="black", linewidth=1.5, label="Mediana"),
    Line2D(
        [0], [0], color="#2f2f2f", marker="o", markerfacecolor="#2f2f2f",
        linewidth=0, markersize=5, alpha=0.45, label="Execucoes"
    ),
    Line2D(
        [0], [0], color="#111111", marker="D", markerfacecolor="white",
        linewidth=0, markersize=6, label="Media"
    ),
]


def plot_metric(metric):
    metric_df = plot_df[plot_df["metric"] == metric.key]
    metric_means = mean_df[mean_df["metric"] == metric.key]
    metric_summary = summary_df[summary_df["metric"] == metric.key]
    fig, ax = plt.subplots(figsize=(11, 5.33), layout="constrained")

    sns.boxplot(
        data=metric_df, x="configuration", y="value", order=configuration_order,
        hue="configuration", hue_order=configuration_order,
        palette=configuration_palette, dodge=False, width=0.62, whis=(0, 100),
        saturation=0.85, linewidth=1.1, fliersize=3,
        medianprops={"color": "#1f1f1f", "linewidth": 1.6},
        whiskerprops={"color": "#555555", "linewidth": 1.1},
        capprops={"color": "#555555", "linewidth": 1.1},
        boxprops={"edgecolor": "#333333"}, ax=ax,
    )
    sns.stripplot(
        data=metric_df, x="configuration", y="value", order=configuration_order,
        color="#2f2f2f", size=3.2, jitter=0.18, alpha=0.45, ax=ax,
    )
    sns.scatterplot(
        data=metric_means, x="configuration", y="value", marker="D", s=42,
        color="white", edgecolor="#111111", linewidth=0.9, zorder=5,
        legend=False, ax=ax,
    )

    ax.set_ylim(0, metric_upper_bound(metric_summary, metric.percent_axis))
    ax.set_title(metric.title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks(range(len(configuration_order)), configuration_labels)
    ax.tick_params(axis="x", rotation=40)
    for tick_label in ax.get_xticklabels():
        tick_label.set_ha("right")
        tick_label.set_rotation_mode("anchor")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.grid(axis="x", visible=False)

    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
    fig.suptitle("Box = Q1-Q3, linha = mediana, pontos = execucoes, losango = media")
    fig.legend(
        handles=stat_legend, loc="lower center", ncols=3,
        bbox_to_anchor=(0.5, -0.01),
    )
    plt.show()

### Média de Pods

In [ ]:
plot_metric(COMPARISON_METRICS[0])

### Limite de CPU

In [ ]:
plot_metric(COMPARISON_METRICS[1])

### Tempo médio das respostas

In [ ]:
plot_metric(COMPARISON_METRICS[2])

### Tamanho médio das respostas

In [ ]:
plot_metric(COMPARISON_METRICS[3])

### Respostas 200

In [ ]:
plot_metric(COMPARISON_METRICS[4])

### Requisições acima do SLO

In [ ]:
plot_metric(COMPARISON_METRICS[5])